# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass **Fine-tuning y Evaluación de Modelos**, se proporciona el siguiente **Colab** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo de lenguaje pequeño y medir el cambio en su comportamiento.

A diferencia de los temas anteriores, aquí no usamos Groq — Groq se utiliza para inferencia, no para entrenar modelos. Usamos **Hugging Face** mediante las librerías transformers, peft y trl directamente sobre la GPU gratuita de Colab.

**Objetivo**

En este ejercicio tomaremos un modelo preentrenado y le enseñaremos un comportamiento específico: responder preguntas de atención a clientes siguiendo las políticas de una tienda ficticia. Compararemos:

**Modelo base → LoRA → entrenamiento → modelo ajustado**

El objetivo no es crear un modelo de producción, sino observar cómo unos pocos ejemplos pueden modificar su comportamiento.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [3]:
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('APIKEY'))
print("Sesión de Hugging Face iniciada correctamente.")

Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usaremos un modelo de lenguaje pequeño para que el fine-tuning pueda ejecutarse en pocos minutos utilizando la GPU gratuita de Google Colab, sin requerir infraestructura especializada.

Para la práctica utilizaremos ***Qwen2.5-0.5B-Instruct***, un modelo instruccional de aproximadamente 500 millones de parámetros. Su tamaño reducido permite realizar el ejercicio en Google Colab y observar de forma práctica cómo LoRA modifica el comportamiento de un modelo preentrenado. *texto en cursiva*

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

modelo_base = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

modelo = AutoModelForCausalLM.from_pretrained(
    modelo_base,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Modelo cargado:", modelo_base)
print("dtype:", next(modelo.parameters()).dtype)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Modelo cargado: Qwen/Qwen2.5-0.5B-Instruct
dtype: torch.float16


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [16]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, pregunta, max_new_tokens=80):

    modelo_a_usar.eval()

    mensajes = [
        {
            "role": "system",
            "content": (
                "Eres un agente de atención al cliente de Tienda Demo. "
                "Responde de forma breve y clara. "
                "No inventes información sobre las políticas de la tienda."

            )
        },
        {
            "role": "user",
            "content": pregunta
        }
    ]

    inputs = tokenizer.apply_chat_template(
        mensajes,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(modelo_a_usar.device)

    with torch.no_grad():
        outputs = modelo_a_usar.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    nuevos_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    respuesta = tokenizer.decode(
        nuevos_tokens,
        skip_special_tokens=True
    )

    return respuesta.strip()

prompt_prueba = "¿Puedo cambiar mi pedido después de pagarlo?"

respuesta_base = generar_respuesta(modelo, prompt_prueba)

print("RESPUESTA DEL MODELO BASE:")
print(respuesta_base)

RESPUESTA DEL MODELO BASE:
Sí, puedes solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com.


In [6]:
prompt_nuevo = (
    "Hice una compra hace media hora y después me di cuenta "
    "de que elegí el artículo equivocado. ¿Qué opciones tengo?"
)
respuesta_base = generar_respuesta(modelo, prompt_nuevo)
print(respuesta_base)

¡Por supuesto! Aquí te presento algunas opciones:

1. **Reembolso**: Si eres del mismo usuario, puedes reembolsar tu compra.
2. **Revisión de Compra**: Puedes revisar tu compra para ver si hay algún error o si es necesario hacer cambios.
3. **Comprueba tu compra**: Si tienes alguna duda sobre


Utilizaremos dos preguntas para comparar el comportamiento del modelo antes y después del fine-tuning.

La primera pregunta aparecerá explícitamente en el dataset de entrenamiento. La segunda expresa la misma intención con una redacción diferente y no aparecerá literalmente en el dataset.

Esto nos permitirá observar posteriormente si el modelo únicamente aprendió los ejemplos de entrenamiento o si puede aplicar la política aprendida ante una formulación diferente.

### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que representen el comportamiento que queremos enseñar al modelo.

En esta demostración utilizaremos 15 ejemplos correspondientes a 5 políticas de atención al cliente, con distintas formas de formular preguntas similares.

El objetivo de estas variaciones es que el modelo no dependa únicamente de una frase exacta, sino que aprenda a relacionar diferentes formulaciones con una misma política.

⚠️ Importante: En un proyecto real normalmente utilizaríamos muchos más ejemplos, mayor diversidad de datos y separaríamos los datos en entrenamiento, validación y prueba.

In [7]:
from datasets import Dataset



SYSTEM_PROMPT = (
    "Eres un agente de atención al cliente de Tienda Demo. "
    "Responde de forma breve y clara. "
    "No inventes información sobre las políticas de la tienda."
)

RESPUESTA_POLITICA = (
    "Sí, puedes solicitar el cambio dentro de la primera hora "
    "escribiendo a soporte@tienda.com."
)

preguntas = [
    "¿Puedo cambiar mi pedido después de pagarlo?",

    "Ya pagué mi pedido, ¿todavía puedo cambiarlo?",

    "¿Es posible modificar una compra que ya pagué?",

    "Me equivoqué en mi pedido y ya lo pagué. ¿Lo puedo cambiar?",

    "Acabo de pagar pero quiero modificar mi pedido. ¿Qué puedo hacer?",

    "¿Cuánto tiempo tengo para cambiar un pedido después de pagarlo?",

    "Hice el pago y noté un error en mi pedido. ¿Todavía puedo corregirlo?",

    "¿Puedo modificar los productos de mi pedido si ya realicé el pago?",

    "Ya confirmé y pagué mi compra, pero necesito hacer un cambio. ¿Es posible?",

    "Pagué hace poco y quiero cambiar uno de los productos de mi pedido. ¿Puedo hacerlo?",

    "Después de pagar, ¿todavía tengo oportunidad de modificar mi compra?",

    "Quiero corregir mi pedido después de haber pagado. ¿Está permitido?",

    "Realicé el pago pero seleccioné un producto incorrecto. ¿Puedo modificar el pedido?",

    "Si ya pagué mi compra, ¿hasta cuándo puedo solicitar un cambio?",

    "Acabo de completar el pago y necesito cambiar mi pedido. ¿Todavía estoy a tiempo?"
]

ejemplos = []

for pregunta in preguntas:
    ejemplos.append({
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": pregunta
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": RESPUESTA_POLITICA
            }
        ]
    })

dataset = Dataset.from_list(ejemplos)

print("Número de ejemplos:", len(dataset))
dataset

Número de ejemplos: 15


Dataset({
    features: ['prompt', 'completion'],
    num_rows: 15
})

Los modelos conversacionales distinguen diferentes roles dentro de una conversación, por ejemplo system, user y assistant.

En nuestro dataset utilizamos el formato conversacional prompt/completion, compatible con SFTTrainer. El prompt contiene las instrucciones del sistema y la pregunta del usuario, mientras que completion contiene la respuesta que queremos que el modelo aprenda.

Durante la inferencia utilizamos apply_chat_template() para presentar las preguntas en el formato de conversación esperado por Qwen.

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [12]:
!pip install "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.6 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [8]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

from peft import LoraConfig, get_peft_model

config_lora = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)

modelo_lora.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [30]:
!pip install -q trl==1.14.2

In [1]:
import trl
print(trl.__version__)

1.14.2


In [9]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning

from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    learning_rate=1e-4,
    logging_steps=1,
    max_length=128,
    completion_only_loss=False,
    loss_type="nll",
    fp16=True,
    bf16=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Step,Training Loss
1,3.795200
2,3.789873
3,3.799318
4,3.619092
5,3.601404
6,3.116159
7,2.693160
8,2.426076
9,2.178910
10,1.832627


Pérdida final: 1.042993148167928


1 epoch significa que el modelo recorrió una vez todo el dataset de entrenamiento.

En esta práctica tenemos 15 ejemplos y utilizaremos 5 épocas, por lo que cada ejemplo será presentado varias veces durante el entrenamiento.

Repetir el dataset ayuda a que el efecto del fine-tuning sea visible en una demostración pequeña, pero un número excesivo de épocas puede provocar sobreajuste (overfitting).

### **DURANTE EL FINE-TUNING: OBSERVAR LA PÉRDIDA**



La pérdida (loss) mide qué tan diferente es la predicción del modelo respecto a los ejemplos que queremos que aprenda.

Una pérdida que disminuye indica que el modelo se está ajustando mejor a los datos de entrenamiento.

Sin embargo, una loss menor no garantiza por sí sola que el modelo generalice correctamente. Por eso también evaluaremos cualitativamente sus respuestas ante una pregunta vista y una pregunta nueva.

In [11]:
# Comparar la pérdidas

losses = [
    registro["loss"]
    for registro in trainer.state.log_history
    if "loss" in registro
]

perdida_inicial = losses[0]
perdida_final = losses[-1]

reduccion = (
    (perdida_inicial - perdida_final)
    / perdida_inicial
) * 100

print(f"Pérdida en el primer paso registrado: {perdida_inicial:.2f}")
print(f"Pérdida en el último paso registrado: {perdida_final:.2f}")
print(f"Reducción: {reduccion:.1f}%")

Pérdida en el primer paso registrado: 3.80
Pérdida en el último paso registrado: 0.23
Reducción: 94.0%


**RESPUESTA DESPUÉS DEL FINE-TUNING**

Compararemos la respuesta después del entrenamiento.

Ahora volvemos a realizar la misma pregunta utilizada antes del entrenamiento.

Nuestro objetivo no es necesariamente obtener una copia exacta de la respuesta del dataset, sino comprobar si el modelo incorporó la política específica de la tienda: los cambios pueden solicitarse únicamente dentro de la primera hora después del pago.

In [12]:
print("DESPUÉS DEL FINE-TUNING:")
print(generar_respuesta(modelo_lora, prompt_prueba))

DESPUÉS DEL FINE-TUNING:
Sí, puedes solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com.


Finalmente utilizamos una pregunta que no apareció literalmente en el dataset de entrenamiento, pero expresa la misma intención.

Si el modelo identifica correctamente que el cambio debe solicitarse dentro de la primera hora, tendremos evidencia de que aprendió a aplicar la política ante una formulación diferente y no únicamente a responder una frase exacta.

Debido al tamaño reducido del dataset, este resultado debe interpretarse como una demostración didáctica y no como una evaluación exhaustiva de generalización.

In [13]:
print("PREGUNTA NUEVA:")
print(generar_respuesta(modelo_lora, prompt_nuevo))

PREGUNTA NUEVA:
Sí, puedes solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com.


Mediante LoRA ajustamos una pequeña fracción de los parámetros utilizando ejemplos de preguntas y respuestas. Después del fine-tuning, el modelo fue capaz de incorporar una regla específica —el límite de una hora para modificar un pedido— y aplicarla tanto a una pregunta vista durante el entrenamiento como a una formulación diferente.

El ejercicio muestra que el fine-tuning puede modificar el comportamiento de un LLM sin necesidad de reentrenar todos sus parámetros. También muestra la importancia de la calidad y diversidad del dataset, los hiperparámetros de entrenamiento y la evaluación con ejemplos no vistos.

Importante: este ejercicio utiliza un dataset extremadamente pequeño con fines educativos. Un sistema de producción requeriría muchos más datos, conjuntos independientes de validación y prueba, métricas de evaluación y controles adicionales para evitar sobreajuste.

In [14]:
print(generar_respuesta(modelo_lora, "¿Aceptan paypal?"))

Sí, aceptamos pagos por PayPal.
